# Supervised Learning: Identifying Crypto Mining Data Centers

Train a classifier to predict the `is_crypto` label from `02-data-cleaning.ipynb`.

In [ ]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import shap
from geopy.distance import geodesic

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.model_selection import cross_validate, cross_val_predict, learning_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score
from sklearn.metrics import precision_recall_curve
from sklearn.inspection import permutation_importance
from sklearn.exceptions import UndefinedMetricWarning

In [ ]:
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")

# display options for pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# some KNN settings predict no crypto in a fold, so precision is undefined there (scored as 0)
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

## Load data

Starting from data/processed/fractracker_data_centers_clean.csv with `is_crypto` label.

In [ ]:
data_centers = pd.read_csv(PROCESSED_DIR / "fractracker_data_centers_clean.csv")

print("Data centers:", data_centers.shape)
print("Data centers labeled crypto:", data_centers["is_crypto"].sum())

### State electricity prices

Add each state's average 2025 industrial electricity price (`Cents/kWh.2`) from the EIA-861M retail sales file.

In [ ]:
sales = pd.read_excel(RAW_DIR / "eia861m_sales_revenue.xlsx", sheet_name="Monthly-States", header=2)
sales_2025 = sales[sales["Year"].astype(str) == "2025"]

# average the 12 months of 2025 for each state
industrial_price = sales_2025.groupby("State")["Cents/kWh.2"].mean()

# one state is stored as "CO " with a trailing space
data_centers["state"] = data_centers["state"].str.strip()
data_centers["industrial_price"] = data_centers["state"].map(industrial_price)

print("Data centers with no price:", data_centers["industrial_price"].isna().sum())
industrial_price.sort_values().head()

# Train a classifier

Following the steps from SIADS 542:
1. Pick the features (`X`) and the label (`is_crypto`).
2. Split into training and test sets.
3. Set a baseline with a dummy classifier.
4. Train logistic regression, a random forest and k-nearest neighbors.
5. Tune each model with `GridSearchCV` and compare them with 5-fold cross-validation.
6. Evaluate the best model and its mistakes.

## Pick the features

Feature groups:
1. **Status**: `status`
2. **Size**: `sizerank`, `mw_high`, `facility_size_sqft`, `property_size_acres`
3. **Missing information**: `has_sqft`, `has_acres`, `has_cost`, `has_buildings`, `has_operator`
4. **Power and cooling**: 0/1 flags from `02-data-cleaning.ipynb` (14 power, 6 cooling)
5. **Community**: `community_pushback`, `resistance_status`
6. **Media coverage**: `n_sources` (count of `info_source_1` to `info_source_8`), `has_other_info`
7. **Electricity price**: `industrial_price`

Preparation:
- One-hot encode `status`, `sizerank` and `resistance_status`.
- Log transform (`np.log1p`) the size numbers. Missing sizes become 0.
- Lowercase `community_pushback` ("Yes" and "yes" both appear).

Left out to prevent leakage:
- `purpose`, `facility_name`, `lat`, `long` and the `crypto_*` columns
- The text of `other_info` and `info_source_*`, `operator_name` and `tenant`

Left out for other reasons: `state`, `mw_low`, `information_source`, `editor`, `location_determination`, `date_created`, `date_updated`.

In [ ]:
X = pd.get_dummies(data_centers[["status", "sizerank"]], dtype=int)

# size: log scale, missing = 0
for col in ["mw_high", "facility_size_sqft", "property_size_acres"]:
    X["log_" + col] = np.log1p(data_centers[col].fillna(0))

# missing information: 1 if FracTracker has the value
X["has_sqft"] = data_centers["facility_size_sqft"].notna().astype(int)
X["has_acres"] = data_centers["property_size_acres"].notna().astype(int)
X["has_cost"] = data_centers["project_cost_low"].notna().astype(int)
X["has_buildings"] = data_centers["buildings_low"].notna().astype(int)
X["has_operator"] = data_centers["operator_name"].notna().astype(int)

# power and cooling: already 0/1 from 02-data-cleaning
power_cols = [
    "batteries", "biomass", "coal", "fuel_cells", "geothermal",
    "grid_unspecified_mix", "hybrid_onsite_and_grid", "hydroelectric",
    "natural_gas", "nuclear", "oil_diesel", "renewable", "solar", "wind",
]
cooling_cols = ["air", "hybrid_air_water", "water", "closed_loop", "fans", "open_loop"]
X[power_cols + cooling_cols] = data_centers[power_cols + cooling_cols]

# community: pushback ("Yes" and "yes" both appear) and resistance status
X["pushback"] = data_centers["community_pushback"].str.lower().eq("yes").astype(int)
resistance = data_centers["resistance_status"].fillna("None")
X = X.join(pd.get_dummies(resistance, prefix="resistance", dtype=int))

# media coverage: count the source links, don't read them
source_cols = ["info_source_" + str(n) for n in range(1, 9)]
X["n_sources"] = data_centers[source_cols].notna().sum(axis=1)
X["has_other_info"] = data_centers["other_info"].notna().astype(int)

# electricity price
X["industrial_price"] = data_centers["industrial_price"]

y = data_centers["is_crypto"]

print("X:", X.shape)
print("y:", y.shape)
X.head()

## Split into training and test sets

Hold back 25% of the rows as a test set. Crypto rows are rare (47 of 1,720, about 2.7%), so `stratify=y` keeps the same share in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=0)

print("Training rows:", len(X_train), "| crypto:", y_train.sum())
print("Test rows:", len(X_test), "| crypto:", y_test.sum())

## Baseline: a dummy classifier

`strategy="most_frequent"` predicts "not crypto" for every row.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

print("Dummy accuracy:", round(dummy.score(X_test, y_test), 3))

The dummy is 97.2% accurate without finding a single crypto data center, so accuracy is not a useful score here. From here on we look at the crypto class:
- **Recall**: of the real crypto data centers, how many did the model find?
- **Precision**: of the data centers the model called crypto, how many really are?

## Logistic regression: first try

Logistic regression needs scaled features, so `MinMaxScaler` puts every feature between 0 and 1. The scaler is fit on the training set only.

In [ ]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train_scaled, y_train)
predictions = model.predict(X_test_scaled)

print("Accuracy:", round(model.score(X_test_scaled, y_test), 3))

# rows = actual (0, 1), columns = predicted (0, 1)
print(confusion_matrix(y_test, predictions))

The model flags only 2 of the 430 test rows as crypto and finds 1 of the 12 crypto data centers. With about 97% of rows labeled 0, always guessing 0 is the easy way to be right.

## Logistic regression: balanced class weights

`class_weight="balanced"` makes a mistake on a rare crypto row cost more, so the model has to try to find them.

In [ ]:
model = LogisticRegression(class_weight="balanced")
model.fit(X_train_scaled, y_train)
predictions = model.predict(X_test_scaled)

print("Accuracy:", round(model.score(X_test_scaled, y_test), 3))

print(confusion_matrix(y_test, predictions))
print()
print(classification_report(y_test, predictions))

Accuracy drops to 0.87, but the model now finds 6 of the 12 crypto data centers.

Positive coefficients push a data center toward crypto, negative ones away from it:

In [ ]:
coefficients = pd.Series(model.coef_[0], index=X.columns)
coefficients.sort_values()

# Non-linear models

Logistic regression draws one straight line between crypto and not crypto. It can't learn combinations like "Operating **and** Large".

## Random forest

A random forest combines many decision trees, each trained on a random sample of rows and features. It takes `class_weight="balanced"` and doesn't need scaled features. We start with 100 trees and `max_depth=3` to limit overfitting. The grid search tunes this later.

In [ ]:
forest = RandomForestClassifier(n_estimators=100, max_depth=3, class_weight="balanced", random_state=0)
forest.fit(X_train, y_train)
predictions = forest.predict(X_test)

print("Accuracy:", round(forest.score(X_test, y_test), 3))

print(confusion_matrix(y_test, predictions))
print()
print(classification_report(y_test, predictions))

The forest finds 8 of the 12 crypto data centers, 2 more than logistic regression, with accuracy 0.86.

`feature_importances_` shows how much each feature helped. The numbers add up to 1:

In [ ]:
importances = pd.Series(forest.feature_importances_, index=X.columns)
importances.sort_values(ascending=False)

## K-nearest neighbors

KNN labels a data center by a majority vote of its `k` most similar data centers in the training set, using the scaled features. It has no `class_weight` option, so with about 97% of rows not crypto, most neighbors vote "not crypto". We start with `n_neighbors=5`.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
predictions = knn.predict(X_test_scaled)

print("Accuracy:", round(knn.score(X_test_scaled, y_test), 3))

print(confusion_matrix(y_test, predictions))
print()
print(classification_report(y_test, predictions, zero_division=0))

KNN barely beats the dummy: it flags only one data center as crypto, so it finds 1 of the 12. The grid search tries other values of `k`.

## Compare the models

Recall and precision for the crypto class on the test set:

In [ ]:
# logistic regression and KNN use the scaled features, the forest doesn't need them
models = {
    "Logistic regression": (model, X_test_scaled),
    "Random forest": (forest, X_test),
    "K-nearest neighbors": (knn, X_test_scaled),
}

rows = []
for name, (m, X_eval) in models.items():
    predictions = m.predict(X_eval)
    rows.append({
        "model": name,
        "recall": round(recall_score(y_test, predictions), 2),
        "precision": round(precision_score(y_test, predictions, zero_division=0), 2),
    })

pd.DataFrame(rows)

Logistic regression and the random forest have similar precision (0.11 and 0.12), and the forest finds 2 more crypto data centers. With only 12 crypto rows in the test set, one split can't tell them apart, so next we tune and compare the models with cross-validation.

# Tune the hyperparameters with GridSearchCV

`GridSearchCV` tries a grid of settings for each model and scores each one with 5-fold cross-validation on the training set:

1. Split the training data into 5 folds.
2. Train on 4 folds and score on the 5th. Repeat for each fold.
3. Take the mean and standard deviation of the 5 scores.

The test set stays untouched.

We pick settings by F1 for the crypto class, which balances precision and recall. Accuracy would reward never guessing crypto.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scoring = ["f1", "recall", "precision", "average_precision"]

# family, model, grid of settings to try
searches = {
    "Logistic regression": (
        "Linear",
        make_pipeline(MinMaxScaler(), LogisticRegression(class_weight="balanced", max_iter=1000)),
        {"logisticregression__C": [0.01, 0.1, 1, 10, 100]},
    ),
    "Random forest": (
        "Tree-based",
        RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=0),
        {"max_depth": [3, 5, 8, None], "min_samples_leaf": [1, 5, 10]},
    ),
    "K-nearest neighbors": (
        "Instance-based",
        make_pipeline(MinMaxScaler(), KNeighborsClassifier()),
        {"kneighborsclassifier__n_neighbors": [1, 3, 5, 7, 9, 15],
         "kneighborsclassifier__weights": ["uniform", "distance"]},
    ),
}

results = {}
for name, (family, m, grid) in searches.items():
    search = GridSearchCV(m, grid, scoring=scoring, refit="f1", cv=cv)
    search.fit(X_train, y_train)
    results[name] = search
    print(name, search.best_params_)

## Compare the tuned models

Best setting for each model: mean ± standard deviation across the 5 folds, plus recall and precision on the test set.

In [ ]:
rows = []
for name, search in results.items():
    best = search.best_index_
    row = {"model": name, "family": searches[name][0]}
    for metric in scoring:
        mean = search.cv_results_["mean_test_" + metric][best]
        std = search.cv_results_["std_test_" + metric][best]
        row["CV " + metric] = f"{mean:.2f} ± {std:.2f}"

    predictions = search.predict(X_test)
    row["test recall"] = round(recall_score(y_test, predictions), 2)
    row["test precision"] = round(precision_score(y_test, predictions, zero_division=0), 2)
    rows.append(row)

cv_summary = pd.DataFrame(rows)
cv_summary

**The random forest is the best model**, with the highest mean F1 (0.32 ± 0.05) and average precision (0.42).
Logistic regression is close on F1 (0.31 ± 0.04), with higher recall (0.69 vs 0.57) but lower precision (0.20 vs 0.23). KNN is last (F1 0.22 ± 0.07).
On the test set the forest's precision is 0.17, about 6 times the 2.7% base rate.

# In-depth evaluation: the random forest

From here on we use the tuned forest (`best_estimator_`), which `GridSearchCV` retrained on the whole training set.

1. Feature importance, permutation importance and ablation
2. Sensitivity analysis
3. Tradeoffs: precision vs recall, and training data size vs F1
4. Failure analysis

Scores use the same 5-fold `cv` as the grid search (mean ± std). Permutation importance, the threshold table and the failure analysis use the test set.

In [ ]:
best_forest = results["Random forest"].best_estimator_
print("Best settings:", results["Random forest"].best_params_)

predictions = best_forest.predict(X_test)

print(confusion_matrix(y_test, predictions))
print()
print(classification_report(y_test, predictions))

## Feature importance

How much each feature helped the forest's splits. The numbers add up to 1. Top 10:

In [ ]:
importances = pd.Series(best_forest.feature_importances_, index=X.columns)
top_10 = importances.sort_values(ascending=False).head(10)
print(top_10)

top_10.sort_values().plot.barh()
plt.title("Random forest: top 10 features")
plt.xlabel("Importance")
plt.show()

**Permutation importance** shuffles one feature at a time in the test set and measures how much F1 drops (10 shuffles, mean ± std). With only 12 crypto rows in the test set, these numbers are noisy.

In [ ]:
result = permutation_importance(best_forest, X_test, y_test, scoring="f1", n_repeats=10, random_state=0)

permutation = pd.DataFrame({
    "mean F1 drop": result.importances_mean,
    "std": result.importances_std,
}, index=X.columns)

permutation.sort_values("mean F1 drop", ascending=False).head(10).round(3)

## Ablation analysis

The two importance measures disagree: the forest ranks power (`log_mw_high`) first, while permutation ranks `industrial_price` and `has_other_info` first. To settle it, we remove one feature group at a time, retrain the forest, and compare the cross-validation F1 (mean ± std). A big drop means the group matters.

In [ ]:
# the feature groups from "Pick the features"
status_cols = [col for col in X.columns if col.startswith("status_")]
size_cols = [col for col in X.columns if col.startswith("sizerank_") or col.startswith("log_")]
resistance_cols = [col for col in X.columns if col.startswith("resistance_")]

feature_groups = {
    "Status": status_cols,
    "Size": size_cols,
    "Missing information": ["has_sqft", "has_acres", "has_cost", "has_buildings", "has_operator"],
    "Power and cooling": power_cols + cooling_cols,
    "Community": ["pushback"] + resistance_cols,
    "Media coverage": ["n_sources", "has_other_info"],
    "Electricity price": ["industrial_price"],
}

# score with all the features first
scores = cross_validate(best_forest, X_train, y_train, cv=cv, scoring="f1")
all_features_f1 = scores["test_score"].mean()

rows = [{
    "removed": "Nothing (all features)",
    "CV f1": round(all_features_f1, 3),
    "std": round(scores["test_score"].std(), 3),
}]

for group_name, group_cols in feature_groups.items():
    X_train_smaller = X_train.drop(columns=group_cols)
    scores = cross_validate(best_forest, X_train_smaller, y_train, cv=cv, scoring="f1")
    rows.append({
        "removed": group_name,
        "CV f1": round(scores["test_score"].mean(), 3),
        "std": round(scores["test_score"].std(), 3),
    })

ablation = pd.DataFrame(rows)
ablation["change"] = (ablation["CV f1"] - all_features_f1).round(3)
ablation

## Sensitivity analysis

Cross-validation F1 for every forest setting the grid search tried. A large std means the score depends on which rows land in each fold.

In [ ]:
grid_results = pd.DataFrame(results["Random forest"].cv_results_)

# max_depth=None (no limit) is stored as a missing value, so we write it out as text
max_depth_labels = []
for depth in grid_results["param_max_depth"]:
    if pd.isna(depth):
        max_depth_labels.append("None")
    else:
        max_depth_labels.append(str(int(depth)))

sensitivity = pd.DataFrame({
    "max_depth": max_depth_labels,
    "min_samples_leaf": grid_results["param_min_samples_leaf"],
    "mean F1": grid_results["mean_test_f1"].round(3),
    "std": grid_results["std_test_f1"].round(3),
})
print(sensitivity.sort_values("mean F1", ascending=False))

# one line per min_samples_leaf ("None" means no depth limit)
for leaf in [1, 5, 10]:
    rows = sensitivity[sensitivity["min_samples_leaf"] == leaf]
    plt.errorbar(rows["max_depth"], rows["mean F1"], yerr=rows["std"], marker="o", capsize=3,
                 label="min_samples_leaf = " + str(leaf))

plt.title("Random forest: F1 for each setting (mean ± std over 5 folds)")
plt.xlabel("max_depth")
plt.ylabel("CV F1")
plt.legend()
plt.show()

## Tradeoff: precision vs recall

`predict` calls a data center crypto when the forest's probability is above 0.5. Lowering the threshold flags more data centers (recall up, precision down). Raising it does the opposite.

In [ ]:
probabilities = best_forest.predict_proba(X_test)[:, 1]

rows = []
for threshold in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    predictions = (probabilities >= threshold).astype(int)
    rows.append({
        "threshold": threshold,
        "flagged as crypto": predictions.sum(),
        "recall": round(recall_score(y_test, predictions), 2),
        "precision": round(precision_score(y_test, predictions, zero_division=0), 2),
    })

print(pd.DataFrame(rows))

precision, recall, thresholds = precision_recall_curve(y_test, probabilities)
plt.plot(recall, precision)
plt.title("Random forest: precision vs recall")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.show()

## Tradeoff: training data size vs F1

`learning_curve` retrains the forest on 12.5%, 25%, 50% and 100% of the training set and scores each with cross-validation. The sizes double each step, so the plot uses a log scale.

In [ ]:
train_sizes, train_scores, validation_scores = learning_curve(
    best_forest, X_train, y_train, cv=cv, scoring="f1",
    train_sizes=[0.125, 0.25, 0.5, 1.0], shuffle=True, random_state=0,
)

learning = pd.DataFrame({
    "training rows": train_sizes,
    "training F1": train_scores.mean(axis=1).round(3),
    "validation F1": validation_scores.mean(axis=1).round(3),
    "validation std": validation_scores.std(axis=1).round(3),
})
print(learning)

plt.plot(train_sizes, train_scores.mean(axis=1), marker="o", label="Training")
plt.errorbar(train_sizes, validation_scores.mean(axis=1), yerr=validation_scores.std(axis=1),
             marker="o", capsize=3, label="Validation (mean ± std)")
plt.xscale("log")
plt.xticks(train_sizes, train_sizes)
plt.title("Random forest: training data size vs F1")
plt.xlabel("Number of training rows (log scale)")
plt.ylabel("F1")
plt.legend()
plt.show()

# Failure analysis

1. Count the false positives (called crypto, labeled not crypto) and false negatives (labeled crypto, missed).
2. List every mistake in a table.
3. Retrain with 20 random seeds to see which mistakes are random.
4. Look closely at one example from three categories, using SHAP.

The tables also show columns the model never saw (`facility_name`, `operator_name`, `purpose`).

In [ ]:
columns_to_show = ["facility_name", "operator_name", "state", "status", "sizerank", "mw_high", "facility_size_sqft", "purpose"]
test_results = data_centers.loc[X_test.index, columns_to_show].copy()

test_results["actual"] = y_test
test_results["predicted"] = best_forest.predict(X_test)
test_results["probability"] = best_forest.predict_proba(X_test)[:, 1].round(2)

false_positives = test_results[(test_results["predicted"] == 1) & (test_results["actual"] == 0)]
false_negatives = test_results[(test_results["predicted"] == 0) & (test_results["actual"] == 1)]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

False positives, most confident first:

In [ ]:
false_positives.sort_values("probability", ascending=False)

False negatives, most confident first (lowest crypto probability):

In [ ]:
false_negatives.sort_values("probability")

## Random or not? Retrain with different seeds

We retrain the forest 20 times, changing only `random_state`, and count how often each test row is wrong. A random mistake should come and go with the seed.

In [ ]:
# same settings as the best forest, only the random seed changes
best_settings = results["Random forest"].best_params_
number_of_seeds = 20

times_wrong = pd.Series(0, index=X_test.index)
for seed in range(number_of_seeds):
    forest_seed = RandomForestClassifier(
        n_estimators=100,
        max_depth=best_settings["max_depth"],
        min_samples_leaf=best_settings["min_samples_leaf"],
        class_weight="balanced",
        random_state=seed,
    )
    forest_seed.fit(X_train, y_train)
    predictions = forest_seed.predict(X_test)
    times_wrong = times_wrong + (predictions != y_test)

share_of_seeds_wrong = times_wrong / number_of_seeds
test_results["share of seeds wrong"] = share_of_seeds_wrong

our_mistakes = test_results["predicted"] != test_results["actual"]
our_correct = test_results["predicted"] == test_results["actual"]
print("Our mistakes that are wrong with every seed:", (share_of_seeds_wrong[our_mistakes] == 1).sum(), "of", our_mistakes.sum())
print("Rows we got right that other seeds sometimes get wrong:", (share_of_seeds_wrong[our_correct] > 0).sum(), "of", our_correct.sum())

## SHAP: why did the forest make each prediction?

A waterfall plot reads from the bottom up. It starts at the forest's average prediction, `E[f(X)]`. Each feature pushes the probability up (red) or down (blue), ending at this row's prediction, `f(x)`. With balanced class weights the average starts near 0.5, not at the 2.7% crypto rate.

In [ ]:
explainer = shap.TreeExplainer(best_forest)
shap_values = explainer(X_test)

# keep the crypto class (class 1)
crypto_shap_values = shap_values[:, :, 1]


def explain_example(facility_name):
    # find the row in the test set
    row_label = test_results[test_results["facility_name"] == facility_name].index[0]
    print(test_results.loc[row_label])

    # SHAP needs the row's position in X_test, not its label
    position = X_test.index.get_loc(row_label)
    shap.plots.waterfall(crypto_shap_values[position], max_display=10)

One example each from three categories: operational, systemic and edge case. The random category has no clear cause to explain. If the data changes, pick new examples from the tables above.

## Example 1: Lake Mariner Data Center (false positive, operational)

The forest is 88% sure this is crypto, but the label says it isn't. The operator, TeraWulf, is a bitcoin miner.

In [ ]:
explain_example("Lake Mariner Data Center")

Is Lake Mariner in the crypto tracker? `01.5-data-aggregation.ipynb` matched crypto sites by address or within 0.075 miles.

In [ ]:
crypto_tracker = pd.read_csv(RAW_DIR / "fractracker_crypto.csv")
tracker_row = crypto_tracker[crypto_tracker["Name"].str.contains("Lake Mariner", na=False)]
print(tracker_row[["Name", "City", "State", "lat", "long"]])

data_center_row = data_centers[data_centers["facility_name"] == "Lake Mariner Data Center"]
print(data_center_row[["facility_name", "city", "state", "lat", "long"]])

tracker_point = (tracker_row["lat"].iloc[0], tracker_row["long"].iloc[0])
data_center_point = (data_center_row["lat"].iloc[0], data_center_row["long"].iloc[0])
print("Distance:", round(geodesic(tracker_point, data_center_point).miles, 3), "miles")

## Example 2: Amazon Madison Mega Site (false positive, systemic)

An Amazon cloud campus, not a crypto mine. The forest is 68% sure it is crypto.

In [ ]:
explain_example("Amazon Madison Mega Site")

Missing building size pushed this row toward crypto. How often is building size missing for each class?

In [ ]:
missing_sqft = data_centers["facility_size_sqft"].isna()

print("Crypto rows missing building size:", round(missing_sqft[data_centers["is_crypto"] == 1].mean(), 2))
print("Other rows missing building size:", round(missing_sqft[data_centers["is_crypto"] == 0].mean(), 2))

## Example 3: Voltcore Bitcoin Mine (false negative, edge case)

A bitcoin mine (its name and `purpose` both say so), but the forest gives it only a 16% chance of being crypto.

In [ ]:
explain_example("Voltcore Bitcoin Mine")

Voltcore's size is `Unknown`. How many crypto rows have an unknown size?

In [ ]:
pd.crosstab(data_centers["sizerank"], data_centers["is_crypto"])